# KGMI — Knowledge Graph Model Intelligence
## From knowledge discovery to model-ready datasets and goal prediction

**Proof of concept · Python / scikit-learn / NetworkX**

### Research question

Can we use machine learning and structured evidence to build a knowledge graph, derive model-consumable datasets from it, and estimate whether a model has the knowledge and capabilities needed to achieve a specified goal?

This notebook demonstrates a small, reproducible version of that workflow:

1. Define a small seed ontology and evidence-backed triples.
2. Construct a typed knowledge graph.
3. Engineer node and relationship features.
4. Use supervised learning to classify candidate relationships.
5. Use unsupervised learning to discover candidate groupings.
6. Compile validated graph facts into triples and instruction-style examples.
7. Calculate goal coverage and identify capability gaps.
8. Train an illustrative goal-success predictor on synthetic historical outcomes.
9. Compare a baseline champion with a graph-augmented challenger on a held-out task set.

### Important scientific boundaries

- A triple `(head, relation, tail)` is a basic knowledge unit, not a complete trustworthy knowledge graph.
- ML predictions are **candidate facts**, not automatically true facts. This notebook keeps inferred edges separate from verified edges.
- Clustering indicates similarity, not proof of a semantic relationship.
- A graph-derived dataset can train or fine-tune a model, but graph embeddings cannot simply be copied into arbitrary LLM weights. Parameter adaptation requires a compatible training objective and architecture.
- The goal score is a coverage measure. The success predictor is a separate model trained on synthetic historical examples; its probabilities are only illustrative and require calibration on real outcomes.
- This notebook does not call an external LLM or train an LLM/LoRA adapter. It demonstrates the data and evaluation interfaces that such an integration would use.

## 1. Environment and imports

The notebook uses common open-source Python packages. If needed, uncomment the installation cell.

In [ ]:
# If running in a fresh environment:
# %pip install -q numpy pandas scikit-learn networkx matplotlib

In [ ]:
import json
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, precision_recall_fscore_support,
    roc_auc_score, brier_score_loss, accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("NetworkX:", nx.__version__)

## 2. Define the goal and seed knowledge

We will use a concrete software/ML engineering goal:

> Perform a defensible ML model security assessment.

The initial knowledge is expressed as triples. Each triple also carries provenance and a status. The seed graph is deliberately small; later steps demonstrate how candidate relationships can be proposed and reviewed.

In [ ]:
seed_triples = [
    ("MLSecurityAssessment", "requires", "PythonSecurity"),
    ("MLSecurityAssessment", "requires", "DependencySecurity"),
    ("MLSecurityAssessment", "requires", "ModelSerializationSecurity"),
    ("MLSecurityAssessment", "requires", "TensorInspection"),
    ("MLSecurityAssessment", "requires", "ArchitectureInspection"),
    ("MLSecurityAssessment", "requires", "DeploymentSecurity"),

    ("PythonSecurity", "uses_tool", "StaticAnalysis"),
    ("DependencySecurity", "uses_tool", "DependencyScanner"),
    ("ModelSerializationSecurity", "uses_tool", "CheckpointInspector"),
    ("TensorInspection", "uses_tool", "TensorInspector"),
    ("ArchitectureInspection", "uses_tool", "ArchitectureParser"),
    ("DeploymentSecurity", "uses_tool", "KubernetesScanner"),

    ("PyTorch", "provides", "TensorOperations"),
    ("PyTorch", "supports", "ModelTraining"),
    ("Transformer", "contains", "SelfAttention"),
    ("Transformer", "contains", "SwiGLU"),
    ("Transformer", "contains", "RMSNorm"),
    ("SelfAttention", "uses", "QKVProjections"),
    ("SelfAttention", "uses", "RoPE"),
    ("ModelCheckpoint", "stores", "ModelParameters"),
    ("Kubernetes", "runs", "ModelServingWorkload"),
]

seed_records = [
    {
        "head": h, "relation": r, "tail": t,
        "source": "seed_domain_spec",
        "status": "verified",
        "confidence": 1.0
    }
    for h, r, t in seed_triples
]

triples_df = pd.DataFrame(seed_records)
triples_df.head(), triples_df.shape

### Triple data contract

A graph fact is represented by `(head, relation, tail)`. Additional metadata records where the fact came from and whether it has been verified.

In a production system, provenance should ideally include a document or code location, timestamp, version, extraction method, and reviewer or validation result.

In [ ]:
def validate_triple(record):
    required = {"head", "relation", "tail", "source", "status", "confidence"}
    missing = required - set(record)
    if missing:
        return False, f"Missing fields: {sorted(missing)}"
    if not all(isinstance(record[k], str) and record[k].strip()
               for k in ("head", "relation", "tail")):
        return False, "Head, relation, and tail must be non-empty strings."
    if not 0.0 <= float(record["confidence"]) <= 1.0:
        return False, "Confidence must be between 0 and 1."
    if record["status"] not in {"verified", "candidate", "rejected"}:
        return False, "Unsupported status."
    return True, "valid"

validation = [validate_triple(r) for r in seed_records]
assert all(ok for ok, _ in validation)
print(f"Validated {len(seed_records)} seed triples.")

## 3. Construct the knowledge graph

We use a directed multigraph so that multiple relationship types can exist between the same entities. Node types are attached as attributes. The graph is a representation of the current knowledge, not a workflow execution DAG.

In [ ]:
G = nx.MultiDiGraph()

# Small, explicit entity type registry for this demonstration.
entity_types = {
    "MLSecurityAssessment": "goal",
    "PythonSecurity": "capability",
    "DependencySecurity": "capability",
    "ModelSerializationSecurity": "capability",
    "TensorInspection": "capability",
    "ArchitectureInspection": "capability",
    "DeploymentSecurity": "capability",
    "StaticAnalysis": "tool",
    "DependencyScanner": "tool",
    "CheckpointInspector": "tool",
    "TensorInspector": "tool",
    "ArchitectureParser": "tool",
    "KubernetesScanner": "tool",
    "PyTorch": "framework",
    "TensorOperations": "concept",
    "ModelTraining": "capability",
    "Transformer": "architecture",
    "SelfAttention": "component",
    "SwiGLU": "component",
    "RMSNorm": "component",
    "QKVProjections": "component",
    "RoPE": "component",
    "ModelCheckpoint": "artifact",
    "ModelParameters": "artifact",
    "Kubernetes": "platform",
    "ModelServingWorkload": "workload",
}

for entity, kind in entity_types.items():
    G.add_node(entity, entity_type=kind)

for row in seed_records:
    G.add_edge(
        row["head"], row["tail"],
        relation=row["relation"],
        source=row["source"],
        status=row["status"],
        confidence=row["confidence"]
    )

print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

In [ ]:
plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=RANDOM_STATE, k=0.9)
nx.draw_networkx_nodes(G, pos, node_size=950)
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edges(G, pos, arrows=True, alpha=0.55, connectionstyle="arc3,rad=0.06")
edge_labels = {(u, v): d["relation"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
plt.title("KGMI seed knowledge graph")
plt.axis("off")
plt.show()

## 4. Feature engineering for nodes

A graph becomes useful to conventional ML when entities and relationships are represented by features. Here we combine simple structural features with entity-type features.

These features are intentionally transparent. Real applications may add source-code AST features, package metadata, text embeddings, static-analysis findings, model metadata, and provenance signals.

In [ ]:
def build_node_feature_frame(graph):
    rows = []
    for node, attrs in graph.nodes(data=True):
        rows.append({
            "node": node,
            "entity_type": attrs.get("entity_type", "unknown"),
            "in_degree": graph.in_degree(node),
            "out_degree": graph.out_degree(node),
            "degree": graph.degree(node),
            "name_length": len(node),
            "has_incoming": int(graph.in_degree(node) > 0),
            "has_outgoing": int(graph.out_degree(node) > 0),
        })
    return pd.DataFrame(rows).set_index("node")

node_features = build_node_feature_frame(G)
node_features.sort_values("degree", ascending=False).head(10)

## 5. Supervised relationship classification

To demonstrate supervised edge classification, we create a small labeled candidate-edge dataset. The labels describe whether a proposed relationship is valid within this toy domain.

**Caution:** these labels are hand-authored demonstration labels, not independently collected ground truth. A production model must be trained and evaluated on representative, independently reviewed examples.

In [ ]:
candidate_edges = [
    ("Transformer", "contains", "SelfAttention", 1),
    ("Transformer", "contains", "SwiGLU", 1),
    ("Transformer", "contains", "RMSNorm", 1),
    ("SelfAttention", "uses", "QKVProjections", 1),
    ("SelfAttention", "uses", "RoPE", 1),
    ("PyTorch", "provides", "TensorOperations", 1),
    ("ModelCheckpoint", "stores", "ModelParameters", 1),
    ("Kubernetes", "runs", "ModelServingWorkload", 1),
    ("PythonSecurity", "uses_tool", "StaticAnalysis", 1),
    ("DependencySecurity", "uses_tool", "DependencyScanner", 1),
    ("Transformer", "uses_tool", "DependencyScanner", 0),
    ("RoPE", "contains", "Transformer", 0),
    ("Kubernetes", "stores", "QKVProjections", 0),
    ("SwiGLU", "runs", "Kubernetes", 0),
    ("ModelParameters", "uses_tool", "PythonSecurity", 0),
    ("PyTorch", "contains", "DependencyScanner", 0),
    ("Attention", "runs", "ModelServingWorkload", 0),
    ("CheckpointInspector", "contains", "RMSNorm", 0),
]

edge_df = pd.DataFrame(
    candidate_edges,
    columns=["head", "relation", "tail", "label"]
)

def edge_features(df):
    out = df.copy()
    out["head_type"] = out["head"].map(entity_types).fillna("unknown")
    out["tail_type"] = out["tail"].map(entity_types).fillna("unknown")
    out["head_name_len"] = out["head"].str.len()
    out["tail_name_len"] = out["tail"].str.len()
    out["head_known"] = out["head"].isin(entity_types).astype(int)
    out["tail_known"] = out["tail"].isin(entity_types).astype(int)
    return out

edge_feature_df = edge_features(edge_df)

In [ ]:
categorical = ["relation", "head_type", "tail_type"]
numeric = ["head_name_len", "tail_name_len", "head_known", "tail_known"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric)
])

edge_classifier = Pipeline([
    ("features", preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced", random_state=RANDOM_STATE))
])

X_edges = edge_feature_df[categorical + numeric]
y_edges = edge_feature_df["label"]

# Tiny illustrative dataset: use a resubstitution fit for demonstration only.
# It is not a valid estimate of generalization performance.
edge_classifier.fit(X_edges, y_edges)
edge_feature_df["predicted_validity"] = edge_classifier.predict_proba(X_edges)[:, 1]
edge_feature_df[["head", "relation", "tail", "label", "predicted_validity"]]

### Candidate edge proposals

The classifier can score proposed relationships. The next cell demonstrates a review queue. A production pipeline should use a held-out evaluation set, class-specific thresholds, evidence checks, and human review before promoting predictions into verified graph facts.

In [ ]:
proposals = pd.DataFrame([
    ("Transformer", "contains", "LayerNorm"),
    ("SelfAttention", "uses", "AttentionMask"),
    ("PyTorch", "supports", "ModelSerialization"),
    ("Kubernetes", "runs", "TrainingJob"),
], columns=["head", "relation", "tail"])

proposal_features = edge_features(proposals)
proposals["model_score"] = edge_classifier.predict_proba(
    proposal_features[categorical + numeric]
)[:, 1]
proposals["status"] = "candidate_only"
proposals.sort_values("model_score", ascending=False)

## 6. Unsupervised discovery: cluster entity representations

We use TF-IDF over entity names and types, then K-means to discover candidate groupings. This is a deliberately simple baseline. In a real system, richer descriptions and domain embeddings would be preferable.

Clusters are hypotheses for exploration, not authoritative ontology labels.

In [ ]:
entity_text = [
    f"{node} {entity_types.get(node, 'unknown')}"
    for node in G.nodes()
]
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
entity_matrix = vectorizer.fit_transform(entity_text)

n_clusters = 4
clusterer = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
cluster_labels = clusterer.fit_predict(entity_matrix)

cluster_df = pd.DataFrame({
    "entity": list(G.nodes()),
    "entity_type": [entity_types.get(n, "unknown") for n in G.nodes()],
    "cluster": cluster_labels
}).sort_values(["cluster", "entity_type", "entity"])

cluster_df

In [ ]:
for cluster_id, group in cluster_df.groupby("cluster"):
    print(f"Cluster {cluster_id}: {', '.join(group['entity'].tolist())}")

## 7. Link prediction by embedding similarity

For a compact demonstration, represent each entity name with TF-IDF and rank similar entities. This is **entity similarity**, not a complete relational link-prediction model such as TransE or a relational GNN.

A stronger implementation would learn entity and relation embeddings from positive triples and carefully generated negative samples.

In [ ]:
similarity = cosine_similarity(entity_matrix)
entity_names = list(G.nodes())
similarity_df = pd.DataFrame(similarity, index=entity_names, columns=entity_names)

def nearest_entities(entity, top_k=5):
    scores = similarity_df.loc[entity].drop(entity).sort_values(ascending=False)
    return scores.head(top_k).rename("similarity").to_frame()

nearest_entities("Transformer")

## 8. Compile graph facts into model-ready datasets

The same validated graph can generate multiple dataset formats:

- Triple dataset for knowledge graph embedding models.
- Text examples for instruction tuning or supervised fine-tuning.
- Candidate query-answer pairs for evaluation.
- Feature tables for conventional classifiers.

We generate examples only from verified edges in this demonstration. Generated examples should retain their source triple IDs so that provenance and train/test leakage can be managed.

In [ ]:
verified_triples = [
    (u, data["relation"], v)
    for u, v, key, data in G.edges(keys=True, data=True)
    if data.get("status") == "verified"
]

triple_dataset = pd.DataFrame(
    verified_triples, columns=["head", "relation", "tail"]
)

instruction_dataset = []
for head, relation, tail in verified_triples:
    instruction_dataset.append({
        "instruction": f"Identify the relationship from {head} to {tail}.",
        "input": f"Entities: {head}, {tail}",
        "output": f"{head} {relation} {tail}.",
        "source_triple": [head, relation, tail]
    })

print("Triple rows:", len(triple_dataset))
print("Instruction examples:", len(instruction_dataset))
pd.DataFrame(instruction_dataset).head()

In [ ]:
# Export artifacts locally. These are synthetic demonstration datasets.
triple_dataset.to_csv("kgmi_triples.csv", index=False)

with open("kgmi_instruction_dataset.jsonl", "w", encoding="utf-8") as f:
    for example in instruction_dataset:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Wrote kgmi_triples.csv and kgmi_instruction_dataset.jsonl")

## 9. Goal specification and graph coverage

A goal is decomposed into required capabilities. Coverage is computed from explicit capability evidence, not inferred from a model's prose response.

For this proof of concept, we define an illustrative capability inventory. In a real system, capabilities should be supported by benchmark results, tool tests, verified artifacts, or other evidence.

In [ ]:
goal_requirements = {
    "PythonSecurity": 0.15,
    "DependencySecurity": 0.15,
    "ModelSerializationSecurity": 0.20,
    "TensorInspection": 0.20,
    "ArchitectureInspection": 0.15,
    "DeploymentSecurity": 0.15,
}

# Demonstration capability assessments for two systems.
capability_profiles = {
    "champion": {
        "PythonSecurity": 0.95,
        "DependencySecurity": 0.90,
        "ModelSerializationSecurity": 0.55,
        "TensorInspection": 0.35,
        "ArchitectureInspection": 0.75,
        "DeploymentSecurity": 0.80,
    },
    "challenger": {
        "PythonSecurity": 0.94,
        "DependencySecurity": 0.91,
        "ModelSerializationSecurity": 0.82,
        "TensorInspection": 0.85,
        "ArchitectureInspection": 0.88,
        "DeploymentSecurity": 0.79,
    }
}

def weighted_coverage(profile, requirements=goal_requirements):
    return sum(
        weight * profile.get(capability, 0.0)
        for capability, weight in requirements.items()
    ) / sum(requirements.values())

coverage_rows = []
for model_name, profile in capability_profiles.items():
    for capability, weight in goal_requirements.items():
        coverage_rows.append({
            "model": model_name,
            "capability": capability,
            "importance": weight,
            "coverage": profile.get(capability, 0.0),
            "weighted_contribution": weight * profile.get(capability, 0.0)
        })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.pivot(index="capability", columns="model", values="coverage")

In [ ]:
coverage_summary = pd.DataFrame([
    {"model": name, "weighted_coverage": weighted_coverage(profile)}
    for name, profile in capability_profiles.items()
])
coverage_summary

In [ ]:
coverage_df.pivot(index="capability", columns="model", values="coverage").plot(
    kind="bar", figsize=(11, 5), ylim=(0, 1), rot=35
)
plt.ylabel("Evidence-based capability score (illustrative)")
plt.title("Champion vs challenger capability coverage")
plt.tight_layout()
plt.show()

## 10. Goal-success prediction from historical outcomes

Coverage is not the same as probability of success. To demonstrate the separate prediction problem, we create a synthetic historical dataset containing capability coverage, input completeness, task complexity, and observed success.

The model learns an association between these features and synthetic outcomes. This is a demonstration of the modeling interface only. It must not be interpreted as a real-world success probability until trained and calibrated on representative real task outcomes.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_history = 2500

history = pd.DataFrame({
    "knowledge_coverage": rng.uniform(0.2, 1.0, n_history),
    "capability_coverage": rng.uniform(0.2, 1.0, n_history),
    "input_completeness": rng.uniform(0.25, 1.0, n_history),
    "task_complexity": rng.uniform(0.0, 1.0, n_history),
    "evidence_quality": rng.uniform(0.2, 1.0, n_history),
})

# Synthetic data-generating process, used only to create demonstration labels.
logit = (
    -3.0
    + 2.0 * history["knowledge_coverage"]
    + 2.5 * history["capability_coverage"]
    + 1.5 * history["input_completeness"]
    + 1.2 * history["evidence_quality"]
    - 2.2 * history["task_complexity"]
)
success_probability = 1 / (1 + np.exp(-logit))
history["success"] = rng.binomial(1, success_probability)

history.head()

In [ ]:
feature_columns = [
    "knowledge_coverage",
    "capability_coverage",
    "input_completeness",
    "task_complexity",
    "evidence_quality"
]

X_train, X_test, y_train, y_test = train_test_split(
    history[feature_columns],
    history["success"],
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=history["success"]
)

goal_predictor = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

goal_predictor.fit(X_train, y_train)
test_probs = goal_predictor.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

print("ROC AUC:", round(roc_auc_score(y_test, test_probs), 3))
print("Brier score:", round(brier_score_loss(y_test, test_probs), 3))
print(classification_report(y_test, test_preds))

In [ ]:
def predict_goal_success(
    knowledge_coverage,
    capability_coverage,
    input_completeness,
    task_complexity,
    evidence_quality
):
    row = pd.DataFrame([{
        "knowledge_coverage": knowledge_coverage,
        "capability_coverage": capability_coverage,
        "input_completeness": input_completeness,
        "task_complexity": task_complexity,
        "evidence_quality": evidence_quality
    }])
    return float(goal_predictor.predict_proba(row)[0, 1])

for name, profile in capability_profiles.items():
    p = predict_goal_success(
        knowledge_coverage=0.80,
        capability_coverage=weighted_coverage(profile),
        input_completeness=0.90,
        task_complexity=0.55,
        evidence_quality=0.85
    )
    print(f"{name}: illustrative predicted success probability = {p:.3f}")

## 11. Champion/challenger comparison

We now compare the two capability profiles using the same synthetic success predictor. This is a controlled demonstration of how a sidecar could rank candidate systems.

In a real experiment, the champion and challenger must be evaluated on the same held-out tasks, with the same inputs, scoring rubric, and tool permissions. The challenger must not be promoted solely because its internal coverage score is higher.

In [ ]:
comparison = []
for name, profile in capability_profiles.items():
    cov = weighted_coverage(profile)
    p = predict_goal_success(
        knowledge_coverage=0.80,
        capability_coverage=cov,
        input_completeness=0.90,
        task_complexity=0.55,
        evidence_quality=0.85
    )
    comparison.append({
        "system": name,
        "weighted_coverage": cov,
        "predicted_success_probability_synthetic": p
    })

comparison_df = pd.DataFrame(comparison)
comparison_df

## 12. What would be required to compile graph knowledge into weights?

This notebook exports graph triples and instruction examples. It does **not** claim that those files are neural weights.

A legitimate parameter-adaptation workflow would be:

1. Freeze and version the validated graph.
2. Generate training examples with provenance.
3. Split by source, entity, or time to prevent leakage.
4. Train a compatible model or LoRA adapter using a defined objective.
5. Evaluate against held-out facts and downstream tasks.
6. Test forgetting, regressions, and safety constraints.
7. Promote the adapter only if it beats the champion under predeclared criteria.

Graph embeddings can also be supplied to a graph-aware model through a learned projection, cross-attention, or retrieval interface. They are not generally shape-compatible with an arbitrary LLM weight matrix.

### Suggested next experiment

Use a small decoder-only PyTorch Transformer as the base model and compare:
- Baseline model.
- Baseline plus graph retrieval.
- Graph-derived supervised fine-tuning or LoRA.
- Graph retrieval plus LoRA.

Measure factual precision, task success, calibration, generalization to unseen entities, latency, and regression on the original benchmark.

## 13. Limitations and production considerations

- **Evidence quality:** Every asserted relationship needs a source and validation status.
- **Open-world assumption:** Missing edges do not necessarily mean a relationship is false.
- **Negative sampling:** Randomly corrupted triples can accidentally create true facts.
- **Data leakage:** Generated paraphrases of the same fact must not cross train/test boundaries.
- **Ontology drift:** Entity types and relation definitions need versioning.
- **Prediction calibration:** A high model score is not a trustworthy probability without calibration on real outcomes.
- **Security:** Model-generated graph updates should be treated as untrusted until validated.
- **Reproducibility:** Record dataset versions, model versions, feature definitions, random seeds, and evaluation criteria.

### Final takeaway

This proof of concept demonstrates a complete conceptual loop:

**Evidence → Knowledge Graph → Features and ML discovery → Model-ready datasets → Goal coverage → Outcome prediction → Champion/challenger evaluation.**

The most important next research step is to replace the synthetic capability profiles and historical outcomes with measured benchmark results from actual models and tasks.